In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
import pandas as pd
from skimage.segmentation import watershed

from tqdm import tqdm

from GDRT.raster_point.raster_point import find_best_shift, corr_nan_weighted_func, WMAE

In [ ]:
FIELD_TREES_FOLDER = "C:\\Users\\david\\dev\\geospatial-data-registration-toolkit\\scratch\\field_trees_shifted\\manual_shifts_filtered"
CHMS_FOLDER = "C:\\Users\\david\\dev\\geospatial-data-registration-toolkit\\scratch\\CHMs"
SHIFTS_PER_DATASET_FILE = "C:\\Users\\david\\dev\\geospatial-data-registration-toolkit\\scratch\\shifts_per_dataset.json"
SHIFT_QUALITY_FILE = "C:\\Users\\david\\dev\\geospatial-data-registration-toolkit\\scratch\\shift_quality.csv"

In [ ]:
shift_qualities = pd.read_csv(SHIFT_QUALITY_FILE)
high_quality_datasets = shift_qualities[shift_qualities["Quality"] > 2].Dataset.values
high_quality_datasets = [d.split(".")[0] for d in high_quality_datasets]

In [ ]:

shifts_per_dataset = json.load(open(SHIFTS_PER_DATASET_FILE))

CHM_files = Path(CHMS_FOLDER).glob("*.tif")
CHM_datasets = [f.stem for f in CHM_files]
matching_datasets = set(list((high_quality_datasets))).intersection(CHM_datasets)

best_shifts = []
ratios = []

for i, dataset in tqdm(enumerate(matching_datasets)):
    field_tree = Path(FIELD_TREES_FOLDER, f"dataset_{dataset}.gpkg")
    CHM = Path(CHMS_FOLDER, f"{dataset}.tif")

    result = find_best_shift(raster_file=CHM, points_file=field_tree, x_range=(-10, 10, 0.5), y_range=(-10, 10, 0.5))

    if not np.all(np.isfinite(result["best_shift"])):
        continue

    img = result["correlations_img"].astype(float)

    first_max = np.nanmax(img)

    max_location_i = np.where(result["y_vals"] == result["best_shift"][1])[0][0]
    max_location_j = np.where(result["x_vals"] == result["best_shift"][0])[0][0]

    seg = watershed(-img, connectivity=2)

    label_of_optimal = seg[max_location_i, max_location_j]

    img[seg == label_of_optimal] = np.nan

    secondary_max = np.nanmax(img)

    ratio = secondary_max / first_max

    best_shifts.append(result["best_shift"])
    ratios.append(ratio)

    if i % 25 == 0:
        plt.imshow(result["correlations_img"])
        plt.xticks(ticks=np.arange(len(result["x_vals"])),labels=result["x_vals"])
        plt.yticks(ticks=np.arange(len(result["y_vals"])),labels=result["y_vals"])
        plt.colorbar()
        plt.title('Correlation surface (unmasked)')
        plt.show()

        plt.imshow(img)
        plt.xticks(ticks=np.arange(len(result["x_vals"])),labels=result["x_vals"])
        plt.yticks(ticks=np.arange(len(result["y_vals"])),labels=result["y_vals"])
        plt.colorbar()
        plt.title('Correlation surface (masked)')
        plt.show()


In [ ]:
best_shifts_array = np.array(best_shifts)

dists = np.linalg.norm(best_shifts_array, axis=1)

plt.hist(dists)
plt.show()
plt.scatter(
    best_shifts_array[:, 0],
    best_shifts_array[:, 1],
    c=ratios,
    cmap="viridis",
)
plt.colorbar()
plt.show()

plt.scatter(ratios, dists)